In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-stage-3-2026")

print("Path to dataset files:", path)

In [ ]:
# i am writing the train path and testing path to pass them to the dataset class
# i got the paths from files bar i checked kaggle file and all linked files path
import os

train_path=os.path.join(path,'PlantVillage', 'train')

test_path=os.path.join(path,'PlantVillage', 'test')

print(train_path)
print(test_path)

In [ ]:
classes=sorted(os.listdir(train_path))
print(classes)

class_names=[]

# i used split to get class names
for names in classes:
  name=names.split('___')[1]
  print(name)
  class_names.append(name)

# i am mapping classes to labels through getting file names because file name represnts class name
class_map={c:i for i,c in enumerate(class_names)}
print(class_map)

print(list(class_map.items()))

In [ ]:
# Write your code here
from PIL import Image

from torch.utils.data import Dataset

class plantvillage(Dataset):
  def __init__(self, path, class_map=class_map, transform=None):
    self.path=path
    self.images_path=[]
    self.labels=[]
    self.class_map=class_map

    for class_folder in os.listdir(self.path):
      c_folder=os.path.join(self.path, class_folder)
      for file_name in os.listdir(c_folder):
        image_path=os.path.join(c_folder, file_name)
        self.images_path.append(image_path)

    def __len__(self):
      return len(self.images_path)

    def __getitem__(self,index):
      image_p=self.images_path[index]
      label=self.class_map[index]
      self.labels.append(label)

      image=Image.open(image_p).convert('RGB')

      if self.transfrom:
        image=self.transform(image)

      return image,label

In [ ]:
import torchvision.transforms as transforms

train_aug=transforms.Compose([
    transforms.Resize((32,32)),
    transforms.RandomRotation(15),
    transforms.ToTensor()
])

test_aug=transforms.Compose([
    transforms.Resize((32,32)),
    transforms.ToTensor()
])

In [ ]:
train_dataset=plantvillage(train_path, class_map=class_map, transform=train_aug)
test_dataset=plantvillage(test_path, class_map=class_map, transform=test_aug)

In [ ]:
from torchvision.datasets import ImageFolder

train_data=ImageFolder(train_path, transform=train_aug)
test_data=ImageFolder(test_path, transform=test_aug)

In [ ]:
print(len(train_data))

In [ ]:
from torch.utils.data import DataLoader

train_loader=DataLoader(train_data, batch_size=32,shuffle=True, num_workers=2)
train_loader=DataLoader(test_data, batch_size=32,shuffle=True, num_workers=2)

In [ ]:
images, label=next(iter(train_loader))

print(images.shape)
print(label[0])

In [ ]:
# Write your code here
import torch.nn as nn
import torch

# Define the CNN Model
class CNNModel(nn.Module):
  def __init__(self):
    super(CNNModel, self).__init__()

    # Convolutional Layers
    self.conv1 = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1)
    self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1)
    self.conv3 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
    self.conv4=nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1)
    self.conv5=nn.Conv2d(in_channels=128, out_channels=256, kernel_size=3, padding=1)

        # Activation
    self.relu = nn.ReLU()

        # Pooling Layer
    self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        # Fully Connected Layers
    self.fc1 = nn.Linear(32, 256)
    self.fc2 = nn.Linear(512, 3)

  def forward(self, x):
      # Convolution + ReLU + Pooling
      x = self.pool(self.relu(self.conv1(x)))
      x = self.pool(self.relu(self.conv2(x)))
      x = self.pool(self.relu(self.conv3(x)))
      x = self.pool(self.relu(self.conv4(x)))
      x = self.pool(self.relu(self.conv5(x)))

        # Flatten
      x = x.view(x.size(0), -1)

        # Fully Connected Layers
      x = self.relu(self.fc1(x))
      x = self.fc2(x)  # Logits

      return x

In [ ]:
# Write your code here


In [ ]:
from tqdm import tqdm    # Shows progress bar

#Training Loop
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()  # Set model to training mode
    total_loss = 0
    correct = 0
    total = 0

    for images, labels in tqdm(dataloader):
        images, labels = images.to(device), labels.to(device)


        outputs = model(images)  # Forward pass
        loss = criterion(outputs, labels)  # Compute loss

        optimizer.zero_grad()  # Reset gradients
        loss.backward()  # Backpropagation
        optimizer.step()  # Update weights

        total_loss += loss.item()

        # Track accuracy
        outputs = torch.softmax(outputs, dim=1)
        predictions = outputs.argmax(dim=1)  # Get class with highest probability
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    avg_loss = total_loss / len(dataloader)
    accuracy = 100 * correct / total  # Compute accuracy in percentage
    return avg_loss, accuracy

# Validation Loop
def validate(model, dataloader, criterion, device):
    model.eval()  # Set model to evaluation mode
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():  # Disable gradient computation
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)  # Forward pass
            loss = criterion(outputs, labels)  # Compute loss
            total_loss += loss.item()

            # Compute accuracy
            outputs = torch.softmax(outputs, dim=1)
            predictions = outputs.argmax(dim=1)  # Get predicted class
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    avg_loss = total_loss / len(dataloader)
    accuracy = 100 * correct / total  # Compute accuracy in percentage
    return avg_loss, accuracy


In [ ]:
# Write your code here




In [ ]:
import torch.optim as optim

# Initialize the model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CNNModel().to(device)

# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()  # Multi-class Classification loss (Input: Logits, not probabilities)
optimizer = optim.Adam(model.parameters(), lr=0.001)  # Adam optimizer
num_epochs = 10 # Number of epochs


# Lists to store metrics
train_losses = []
val_losses = []
train_accuracies = []
val_accuracies = []

# Training process
for epoch in range(num_epochs):
    train_loss, train_accuracy = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_accuracy = validate(model, test_loader, criterion, device)

    # Store metrics
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accuracies.append(train_accuracy)
    val_accuracies.append(val_accuracy)

    print(f"Epoch {epoch+1}/{num_epochs}: "
          f"Train Loss={train_loss:.4f}, Train Accuracy={train_accuracy:.2f}%, "
          f"Val Loss={val_loss:.4f}, Val Accuracy={val_accuracy:.2f}%")


In [ ]:
# Write your code here
